# Notebook 03 · Dataset E0 — Baseline sin Augmentación Sintética

Este notebook construye el escenario experimental **E0** para el fine-tuning
de los modelos *RoBERTa-BNE* y *MarIA* sobre el corpus FakeDeS.
E0 constituye la línea base del estudio: utiliza exclusivamente los ejemplos
originales del corpus, sin incorporar texto sintético generado en HU09.

| Artefacto de salida | Ruta |
|---|---|
| Split de entrenamiento | `data/processed/E0/train.csv` |
| Split de validación | `data/processed/E0/val.csv` |
| Split de prueba | `data/processed/E0/test.csv` |
| Ficha de trazabilidad | `data/processed/E0/metadata_E0.json` |

## § 0 · Dependencias

Importaciones necesarias para la construcción, serialización y verificación
del dataset E0.

In [3]:
import warnings
from pathlib import Path
import pandas as pd
import json
from datetime import datetime, timezone
import hashlib
warnings.filterwarnings('ignore')

## § 1 · Configuración y Carga de Datos

E0 es el escenario **baseline** del experimento: reproduce exactamente la
partición estratificada 80/10/10 generada en el Notebook 01, sin ninguna
modificación del contenido textual ni incorporación de ejemplos sintéticos.
Este escenario sirve como referencia para cuantificar el impacto de la
augmentación sintética (E1) en las métricas de clasificación de los modelos
*RoBERTa-BNE* (HU14) y *MarIA* (HU16).

In [5]:
ROOT_DIR      = Path().resolve().parent
PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
E0_DIR        = PROCESSED_DIR / 'E0'
E0_DIR.mkdir(parents=True, exist_ok=True)

# Carga de los splits producidos por el Notebook 01
df_train = pd.read_csv(PROCESSED_DIR / 'train.csv', encoding='utf-8')
df_val   = pd.read_csv(PROCESSED_DIR / 'val.csv',   encoding='utf-8')
df_test  = pd.read_csv(PROCESSED_DIR / 'test.csv',  encoding='utf-8')

# Resumen de shapes y distribución de clases por split
for nombre, df in [('train', df_train), ('val', df_val), ('test', df_test)]:
    n_fake = int((df['Category'] == 1).sum())
    n_true = int((df['Category'] == 0).sum())
    print(f'{nombre:<6}  shape={df.shape}  '
          f'fake(1)={n_fake}  true(0)={n_true}  '
          f'%fake={n_fake/len(df)*100:.1f}')

train   shape=(772, 7)  fake(1)=381  true(0)=391  %fake=49.4
val     shape=(97, 7)  fake(1)=48  true(0)=49  %fake=49.5
test    shape=(97, 7)  fake(1)=48  true(0)=49  %fake=49.5


## § 2 · Construcción del Dataset E0

Se seleccionan únicamente las columnas `Text` y `Category` para cada split.
Las filas con texto nulo o cadena vacía tras el strip se eliminan para
garantizar que el tokenizador no reciba entradas degeneradas durante el
fine-tuning.

In [7]:
# Selección de columnas y copia defensiva
df_train_E0 = df_train[['Text', 'Category']].copy()
df_val_E0   = df_val[['Text', 'Category']].copy()
df_test_E0  = df_test[['Text', 'Category']].copy()

# Eliminación de filas con Text nulo o vacío
for df in (df_train_E0, df_val_E0, df_test_E0):
    mask_nulo  = df['Text'].isna()
    mask_vacio = df['Text'].astype(str).str.strip() == ''
    df.drop(index=df.index[mask_nulo | mask_vacio], inplace=True)
    df.reset_index(drop=True, inplace=True)

# Tabla resumen
header = f"{'Split':<6} {'Total':>7} {'Fake(1)':>8} {'True(0)':>8} {'%Fake':>7}"
sep    = '-' * len(header)
print(header)
print(sep)
for nombre, df in [('train', df_train_E0), ('val', df_val_E0), ('test', df_test_E0)]:
    n_total = len(df)
    n_fake  = int((df['Category'] == 1).sum())
    n_true  = int((df['Category'] == 0).sum())
    pct     = n_fake / n_total * 100 if n_total else 0.0
    print(f'{nombre:<6} {n_total:>7} {n_fake:>8} {n_true:>8} {pct:>6.1f}%')

Split    Total  Fake(1)  True(0)   %Fake
----------------------------------------
train      772      381      391   49.4%
val         97       48       49   49.5%
test        97       48       49   49.5%


## § 3 · Guardado de Archivos

Los tres splits se persisten en el directorio `data/processed/E0/`.
El hash MD5 de cada archivo se calcula sobre el contenido binario
del fichero ya escrito, de modo que la ficha de trazabilidad refleja
el estado exacto del artefacto en disco.

In [9]:
def _md5(path: Path) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

splits = [
    ('train', df_train_E0, E0_DIR / 'train.csv'),
    ('val',   df_val_E0,   E0_DIR / 'val.csv'),
    ('test',  df_test_E0,  E0_DIR / 'test.csv'),
]

hashes = {}
for nombre, df, ruta in splits:
    df.to_csv(ruta, index=False, encoding='utf-8')
    hashes[nombre] = _md5(ruta)
    print(f'{nombre:<6}  registros={len(df):>6}  md5={hashes[nombre]}')

train   registros=   772  md5=a6df6b31ce9895550878a7f339b12117
val     registros=    97  md5=2d22860c2f6cbfeccca34a75f1d356ff
test    registros=    97  md5=06697b127fd01b181ff5e50d87a5a2d3


## § 4 · Ficha de Trazabilidad E0

La ficha `metadata_E0.json` consolida los parámetros de configuración,
las estadísticas descriptivas de cada split y los hashes MD5 de los
artefactos en disco. Este registro garantiza la reproducibilidad del
escenario E0 en los experimentos de fine-tuning subsiguientes.

In [11]:
n_fake_train = int((df_train_E0['Category'] == 1).sum())
n_true_train = int((df_train_E0['Category'] == 0).sum())
prop_fake    = round(n_fake_train / len(df_train_E0), 4) if len(df_train_E0) else 0.0

metadata = {
    'escenario':             'E0',
    'descripcion':           'Corpus FakeDeS sin augmentación sintética. Escenario baseline.',
    'fecha_construccion':    datetime.now(timezone.utc).isoformat(),
    'semilla_particion':     42,
    'fuente_train':          'data/processed/train.csv',
    'fuente_val':            'data/processed/val.csv',
    'fuente_test':           'data/processed/test.csv',
    'n_train':               len(df_train_E0),
    'n_val':                 len(df_val_E0),
    'n_test':                len(df_test_E0),
    'n_fake_train':          n_fake_train,
    'n_true_train':          n_true_train,
    'proporcion_fake_train': prop_fake,
    'hash_md5_train':        hashes['train'],
    'hash_md5_val':          hashes['val'],
    'hash_md5_test':         hashes['test'],
    'columnas':              ['Text', 'Category'],
    'modelos_objetivo':      ['RoBERTa-BNE', 'MarIA'],
    'augmentacion_aplicada': False,
}

META_PATH = E0_DIR / 'metadata_E0.json'
with open(META_PATH, 'w', encoding='utf-8') as fh:
    json.dump(metadata, fh, ensure_ascii=False, indent=2)

print(json.dumps(metadata, ensure_ascii=False, indent=2))

{
  "escenario": "E0",
  "descripcion": "Corpus FakeDeS sin augmentación sintética. Escenario baseline.",
  "fecha_construccion": "2026-06-04T19:34:20.577029+00:00",
  "semilla_particion": 42,
  "fuente_train": "data/processed/train.csv",
  "fuente_val": "data/processed/val.csv",
  "fuente_test": "data/processed/test.csv",
  "n_train": 772,
  "n_val": 97,
  "n_test": 97,
  "n_fake_train": 381,
  "n_true_train": 391,
  "proporcion_fake_train": 0.4935,
  "hash_md5_train": "a6df6b31ce9895550878a7f339b12117",
  "hash_md5_val": "2d22860c2f6cbfeccca34a75f1d356ff",
  "hash_md5_test": "06697b127fd01b181ff5e50d87a5a2d3",
  "columnas": [
    "Text",
    "Category"
  ],
  "modelos_objetivo": [
    "RoBERTa-BNE",
    "MarIA"
  ],
  "augmentacion_aplicada": false
}


## § 5 · Verificación Final

Los tres archivos CSV se recargan desde disco para confirmar que los
hashes MD5 calculados tras la lectura coinciden con los registrados en
`metadata_E0.json`. La verificación cierra el ciclo de integridad del
escenario E0.

In [13]:
# Recarga desde disco
df_train_v = pd.read_csv(E0_DIR / 'train.csv', encoding='utf-8')
df_val_v   = pd.read_csv(E0_DIR / 'val.csv',   encoding='utf-8')
df_test_v  = pd.read_csv(E0_DIR / 'test.csv',  encoding='utf-8')

# Recalculo de hashes
hashes_v = {
    'train': _md5(E0_DIR / 'train.csv'),
    'val':   _md5(E0_DIR / 'val.csv'),
    'test':  _md5(E0_DIR / 'test.csv'),
}

# Tabla de verificación
header = f"{'Archivo':<20} {'Registros':>10} {'Hash MD5':<34} {'Estado':>6}"
sep    = '-' * len(header)
print(header)
print(sep)

filas = [
    ('E0/train.csv', df_train_v, 'train'),
    ('E0/val.csv',   df_val_v,   'val'),
    ('E0/test.csv',  df_test_v,  'test'),
]
todo_ok = True
for archivo, df_v, clave in filas:
    hash_meta = metadata[f'hash_md5_{clave}']
    hash_disk = hashes_v[clave]
    estado    = 'OK' if hash_meta == hash_disk else 'ERROR'
    if estado == 'ERROR':
        todo_ok = False
    print(f'{archivo:<20} {len(df_v):>10} {hash_disk:<34} {estado:>6}')

print()
if todo_ok:
    print('Dataset E0 verificado y listo para fine-tuning.')
else:
    raise RuntimeError('Se detectaron discrepancias en los hashes MD5.')

Archivo               Registros Hash MD5                           Estado
-------------------------------------------------------------------------
E0/train.csv                772 a6df6b31ce9895550878a7f339b12117       OK
E0/val.csv                   97 2d22860c2f6cbfeccca34a75f1d356ff       OK
E0/test.csv                  97 06697b127fd01b181ff5e50d87a5a2d3       OK

Dataset E0 verificado y listo para fine-tuning.
